# Fase 2: Pipeline de Procesamiento de Lenguaje Natural (PLN)
## Tratamiento de Narrativas y Entrenamiento del Clasificador de Dramatismo

Este notebook documenta la **Fase 2 del sistema**, que transforma datos textuales no estructurados (prensa, redes sociales, narraciones minuto a minuto) en señales cuantitativas calibradas para el motor de recomendación:
1. **Análisis de Sentimiento y Hype**: Extracción de polaridad con NLTK VADER y cálculo del *Fan Excitement Index* $[0, 100]$.
2. **Embeddings Semánticos Multieje**: Proyección vectorial en 4 dimensiones narrativas (`title_race`, `derby_rivalry`, `relegation_scrap`, `attacking_chaos`) con TF-IDF y similitud de cosenos.
3. **Métrica Léxica de Rivalidad y Derbi**: Evaluación de hostilidad histórica y tensión deportiva.
4. **Entrenamiento y Evaluación del Clasificador de Dramatismo Narrativo (`CommentaryDramaClassifier`)**:
   - Entrenamiento supervisado con `LogisticRegression` sobre n-gramas TF-IDF de comentarios minuto a minuto.
   - Matriz de confusión, reporte de clasificación, coeficientes léxicos más predictivos y moduladores contextuales.
5. **Extracción de Trending Keywords y Score PLN Normalizado**.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import nltk

BASE_DIR = Path(os.path.abspath("..")) if Path("..").resolve().name == "sistema_de_recomendaciones" else Path(os.path.abspath("."))
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src.core.config import settings
from src.data_ingestion.local_provider import LocalDataProvider
from src.features.nlp import ContextSemanticEmbedder, NLPFeatureExtractor
from src.features.commentary_classifier import CommentaryDramaClassifier

from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

print("✔ Dependencias de PLN importadas exitosamente.")
print(f"✔ Ruta de NLTK Data: {settings.nltk_data_dir}")

✔ Dependencias de PLN importadas exitosamente.
✔ Ruta de NLTK Data: /Users/hugoportillo/sistema_de_recomendaciones/.venv/share/nltk_data

---
## 1. Tratamiento de Sentimiento y Hype de Aficionados (Reddit) con NLTK VADER

NLTK VADER (*Valence Aware Dictionary and sEntiment Reasoner*) está calibrado específicamente para texto informal de redes sociales.
A diferencia de textos formales, en el deporte un partido con alto "hype" puede incluir lenguaje intenso o visceral. Por ello, el **Fan Excitement Index** integra:
- La polaridad base normalizada: $\frac{compound + 1}{2} \times 50$
- La carga emocional neta: $|compound| \times 30$
- La velocidad de comentarios por minuto: $\min(20, \frac{vel}{6} \times 20)$
- Bonificación adicional de $+10$ puntos si el partido es un derbi oficial.

In [ ]:
sia = SentimentIntensityAnalyzer()
local_provider = LocalDataProvider(data_dir=settings.data_dir)
fixtures = local_provider.get_fixtures(gameweek=29)

sentiment_records = []
for f in fixtures:
    narrative = local_provider.get_narratives(f.id)
    comments = narrative.get("fan_comments", [])
    
    compounds = [sia.polarity_scores(c)["compound"] for c in comments] if comments else [0.0]
    avg_compound = float(np.mean(compounds))
    
    # Simulación de velocidad de comentarios por minuto
    comment_velocity = 5.8 if f.is_derby else (4.5 if f.home_team.position <= 4 else 2.2)
    
    # Cálculo del Fan Excitement Index
    base_sentiment = ((avg_compound + 1.0) / 2.0) * 50.0
    emotional_intensity = abs(avg_compound) * 30.0
    velocity_boost = min(20.0, (comment_velocity / 6.0) * 20.0)
    fan_excitement = min(100.0, max(10.0, base_sentiment + emotional_intensity + velocity_boost))
    if f.is_derby:
        fan_excitement = min(100.0, fan_excitement + 10.0)
        
    sentiment_records.append({
        "fixture_id": f.id,
        "match": f"{f.home_team.short_name} vs {f.away_team.short_name}",
        "is_derby": f.is_derby,
        "sample_comment": comments[0] if comments else "N/A",
        "avg_compound": round(avg_compound, 3),
        "comment_velocity": comment_velocity,
        "fan_excitement_index": round(fan_excitement, 1)
    })

df_sentiment = pd.DataFrame(sentiment_records).sort_values("fan_excitement_index", ascending=False).reset_index(drop=True)
print("--- FAN EXCITEMENT INDEX (VADER + HYPE AFICIÓN) ---")
print(df_sentiment[["match", "is_derby", "avg_compound", "comment_velocity", "fan_excitement_index"]].to_string(index=False))

--- FAN EXCITEMENT INDEX (VADER + HYPE AFICIÓN) ---
     match  is_derby  avg_compound  comment_velocity  fan_excitement_index
AVL vs WOL      True         0.406               5.8                  76.7
BOU vs SHU     False         0.610               2.2                  65.9
MCI vs ARS     False         0.426               4.5                  63.4
TOT vs CHE      True        -0.304               5.8                  55.9
BHA vs CRY      True        -0.155               5.8                  55.1
LIV vs EVE      True        -0.029               5.8                  54.5
NEW vs WHU     False         0.237               2.2                  45.4
LUT vs NFO     False        -0.430               2.2                  34.5
BUR vs BRE     False        -0.328               2.2                  34.0
MUN vs FUL     False        -0.099               2.2                  32.8

---
## 2. Embeddings Semánticos Multieje (4 Ejes Narrativos)

Para clasificar la temática de cada previa periodística y conversación social, proyectamos los textos sobre 4 ejes semánticos canónicos:
1. `title_race`: Lucha directa por el título y la cima de la tabla.
2. `derby_rivalry`: Rivalidad local, hostilidad deportiva y orgullo barrial.
3. `relegation_scrap`: Angustia competitiva por evitar el descenso.
4. `attacking_chaos`: Partidos dinámicos de ida y vuelta con alto volumen ofensivo.

Utilizamos `ContextSemanticEmbedder`, que opera en modo Zero-Cost mediante vectorización TF-IDF con n-gramas `(1, 2)` y similitud de cosenos.

In [ ]:
embedder = ContextSemanticEmbedder()

narrative_embeddings = []
for f in fixtures:
    narrative = local_provider.get_narratives(f.id)
    all_text = " ".join(narrative.get("headlines", []) + narrative.get("fan_comments", []))
    
    affinity = embedder.embed_and_score(all_text)
    narrative_embeddings.append({
        "fixture_id": f.id,
        "match": f"{f.home_team.name} vs {f.away_team.name}",
        "title_race": affinity.get("title_race", 0.0),
        "derby_rivalry": affinity.get("derby_rivalry", 0.0),
        "relegation_scrap": affinity.get("relegation_scrap", 0.0),
        "attacking_chaos": affinity.get("attacking_chaos", 0.0),
        "dominant_axis": max(affinity, key=affinity.get)
    })

df_embeddings = pd.DataFrame(narrative_embeddings)
print("--- AFINIDAD CON EJES NARRATIVOS (EMBEDDINGS TF-IDF + COSENOS) ---")
print(df_embeddings[["match", "title_race", "derby_rivalry", "relegation_scrap", "attacking_chaos", "dominant_axis"]].to_string(index=False))

--- AFINIDAD CON EJES NARRATIVOS (EMBEDDINGS TF-IDF + COSENOS) ---
                                   match  title_race  derby_rivalry  relegation_scrap  attacking_chaos    dominant_axis
              Manchester City vs Arsenal       0.480          0.000             0.042            0.186       title_race
                    Liverpool vs Everton       0.046          0.383             0.144            0.085    derby_rivalry
            Tottenham Hotspur vs Chelsea       0.044          0.284             0.091            0.284    derby_rivalry
     Newcastle United vs West Ham United       0.115          0.000             0.120            0.107 relegation_scrap
  Aston Villa vs Wolverhampton Wanderers       0.000          0.392             0.000            0.000    derby_rivalry
         Luton Town vs Nottingham Forest       0.000          0.000             0.503            0.056 relegation_scrap
Brighton & Hove Albion vs Crystal Palace       0.000          0.281             0.000        

---
## 3. Entrenamiento y Evaluación del Clasificador de Dramatismo Narrativo (`CommentaryDramaClassifier`)

El **Clasificador de Dramatismo Narrativo** predice si el historial narrativo y las crónicas previas sugieren un partido propenso a desenlaces épicos, remontadas agónicas y alta fricción (Etiqueta 1) frente a partidos de trámite controlado y bajo ritmo (Etiqueta 0).

### Pipeline de Modelado:
- **Corpus**: Extractos de narraciones minuto a minuto y crónicas históricas.
- **Extracción de Características**: `TfidfVectorizer(ngram_range=(1, 2), stop_words="english")`.
- **Algoritmo**: `LogisticRegression(C=1.5, solver="lbfgs")`.
- **Moduladores Contextuales**:
  $$\Delta_{derby} = +0.15 \quad \text{si es derbi oficial}$$
  $$\Delta_{tabla} = +0.08 \quad \text{si } |Pos_L - Pos_V| \le 2$$

In [ ]:
# 1. Definición del Corpus Histórico de Entrenamiento
training_samples = CommentaryDramaClassifier.TRAINING_SAMPLES
texts, labels = zip(*training_samples)

print(f"Total de muestras de entrenamiento: {len(texts)}")
print(f"Distribución de clases: {sum(labels)} partidos Dramáticos (1) vs {len(labels) - sum(labels)} Rutinarios (0)")

# 2. Vectorización TF-IDF con n-gramas (unigramas y bigramas)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
X_train = vectorizer.fit_transform(texts)
y_train = np.array(labels)

print(f"Dimensión del espacio vectorial TF-IDF: {X_train.shape[1]} características (términos y bigramas)")

# 3. Entrenamiento del Modelo de Regresión Logística
model = LogisticRegression(C=1.5, solver="lbfgs", random_state=42)
model.fit(X_train, y_train)

# 4. Evaluación del Modelo sobre el conjunto de entrenamiento
y_pred = model.predict(X_train)
acc = accuracy_score(y_train, y_pred)
conf_mat = confusion_matrix(y_train, y_pred)

print("\n--- EVALUACIÓN DEL CLASIFICADOR DE DRAMATISMO ---")
print(f"Accuracy del Modelo: {acc * 100:.1f}%")
print("\nMatriz de Confusión:")
print(conf_mat)

print("\nReporte de Clasificación:")
print(classification_report(y_train, y_pred, target_names=["Rutinario / Bajo Drama", "Alto Dramatismo"]))

Total de muestras de entrenamiento: 13
Distribución de clases: 7 partidos Dramáticos (1) vs 6 Rutinarios (0)
Dimensión del espacio vectorial TF-IDF: 238 características (términos y bigramas)

--- EVALUACIÓN DEL CLASIFICADOR DE DRAMATISMO ---
Accuracy del Modelo: 100.0%

Matriz de Confusión:
[[6 0]
 [0 7]]

Reporte de Clasificación:
                        precision    recall  f1-score   support

Rutinario / Bajo Drama       1.00      1.00      1.00         6
       Alto Dramatismo       1.00      1.00      1.00         7

              accuracy                           1.00        13
             macro avg       1.00      1.00      1.00        13
          weighted avg       1.00      1.00      1.00        13


### Interpretabilidad del Modelo: Coeficientes Léxicos de Dramatismo

Examinamos qué palabras y combinaciones de n-gramas aportan mayor peso predictivo hacia partidos de alto dramatismo vs partidos rutinarios.

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = model.coef_[0]

# Ordenar coeficientes
top_drama_indices = np.argsort(coefficients)[::-1][:8]
top_routine_indices = np.argsort(coefficients)[:8]

df_vocab_drama = pd.DataFrame({
    "Término / Bigrama": feature_names[top_drama_indices],
    "Coeficiente Logit": np.round(coefficients[top_drama_indices], 3),
    "Impacto": "Alto Dramatismo ⚡"
})

df_vocab_routine = pd.DataFrame({
    "Término / Bigrama": feature_names[top_routine_indices],
    "Coeficiente Logit": np.round(coefficients[top_routine_indices], 3),
    "Impacto": "Trámite Rutinario 🛡️"
})

print("🔥 TOP TÉRMINOS ASOCIADOS A ALTO DRAMATISMO:")
print(df_vocab_drama.to_string(index=False))

print("\n🧊 TOP TÉRMINOS ASOCIADOS A BAJO DRAMATISMO:")
print(df_vocab_routine.to_string(index=False))

🔥 TOP TÉRMINOS ASOCIADOS A ALTO DRAMATISMO:
Término / Bigrama  Coeficiente Logit           Impacto
              end              0.227 Alto Dramatismo ⚡
            goals              0.217 Alto Dramatismo ⚡
             pure              0.198 Alto Dramatismo ⚡
            chaos              0.198 Alto Dramatismo ⚡
         stoppage              0.179 Alto Dramatismo ⚡
    stoppage time              0.179 Alto Dramatismo ⚡
         dramatic              0.179 Alto Dramatismo ⚡
             time              0.179 Alto Dramatismo ⚡

🧊 TOP TÉRMINOS ASOCIADOS A BAJO DRAMATISMO:
Término / Bigrama  Coeficiente Logit              Impacto
       possession             -0.245 Trámite Rutinario 🛡️
            tempo             -0.233 Trámite Rutinario 🛡️
            quiet             -0.232 Trámite Rutinario 🛡️
  points breaking             -0.152 Trámite Rutinario 🛡️
    sided contest             -0.152 Trámite Rutinario 🛡️
predictable sided             -0.152 Trámite Rutinario 🛡️
   breakin

### Inferencia y Moduladores Contextuales en la Jornada 29

Aplicamos el clasificador de dramatismo entrenado (`predict_drama_score`) sobre todos los partidos de la jornada 29, incorporando los moduladores contextuales de derbi y diferencia en la tabla clasificatoria.

In [ ]:
drama_classifier = CommentaryDramaClassifier()

drama_scores = []
for f in fixtures:
    narrative = local_provider.get_narratives(f.id)
    all_text = " ".join(narrative.get("headlines", []) + narrative.get("fan_comments", []))
    pos_diff = abs(f.home_team.position - f.away_team.position)
    
    score = drama_classifier.predict_drama_score(
        full_text=all_text,
        is_derby=f.is_derby,
        table_diff=pos_diff
    )
    
    drama_scores.append({
        "fixture_id": f.id,
        "match": f"{f.home_team.name} vs {f.away_team.name}",
        "is_derby": f.is_derby,
        "pos_diff": pos_diff,
        "drama_probability": score,
        "dramatism_pct": f"{int(score * 100)}%"
    })

df_drama = pd.DataFrame(drama_scores).sort_values("drama_probability", ascending=False).reset_index(drop=True)
print("--- PROBABILIDAD DE DRAMATISMO NARRATIVO POR PARTIDO ---")
print(df_drama[["match", "is_derby", "pos_diff", "drama_probability", "dramatism_pct"]].to_string(index=False))

--- PROBABILIDAD DE DRAMATISMO NARRATIVO POR PARTIDO ---
                                   match  is_derby  pos_diff  drama_probability dramatism_pct
            Tottenham Hotspur vs Chelsea      True         6              0.792           79%
                    Liverpool vs Everton      True        14              0.733           73%
  Aston Villa vs Wolverhampton Wanderers      True         5              0.706           70%
              Manchester City vs Arsenal     False         2              0.704           70%
         Luton Town vs Nottingham Forest     False         1              0.696           69%
Brighton & Hove Albion vs Crystal Palace      True         6              0.681           68%
             Manchester United vs Fulham     False         6              0.583           58%
     Newcastle United vs West Ham United     False         3              0.551           55%
                    Burnley vs Brentford     False         4              0.531           53%
   

---
## 4. Score Cualitativo de PLN Consolidado (`MatchNLPFeatures`)

Combinamos las 4 dimensiones cualitativas del pipeline de PLN:
- 35% Fan Excitement Index
- 25% Aceleración Mediática (Media Velocity)
- 20% Rivalidad Narrativa y Derbi (TF-IDF Cosine)
- 20% Probabilidad de Dramatismo Narrativo

$$Score_{PLN} = 0.35 \cdot Fan + 0.25 \cdot Media + 0.20 \cdot Rivalidad + 0.20 \cdot Drama$$

In [ ]:
nlp_extractor = NLPFeatureExtractor()

nlp_features_list = []
for f in fixtures:
    narrative = local_provider.get_narratives(f.id)
    nlp_feat = nlp_extractor.extract(f, narrative)
    
    nlp_features_list.append({
        "fixture_id": f.id,
        "match": f"{f.home_team.short_name} vs {f.away_team.short_name}",
        "fan_excitement": nlp_feat.fan_excitement_index,
        "media_velocity": nlp_feat.media_velocity,
        "rivalry_score": nlp_feat.narrative_rivalry_score,
        "drama_score": nlp_feat.dramatic_intensity_score,
        "trending_keywords": ", ".join(nlp_feat.trending_keywords[:3]),
        "normalized_nlp_score": nlp_feat.normalized_nlp_score
    })

df_nlp_final = pd.DataFrame(nlp_features_list).sort_values("normalized_nlp_score", ascending=False).reset_index(drop=True)
print("--- RANKING DE PARTIDOS POR SCORE CUALITATIVO DE PLN [0-100] ---")
print(df_nlp_final[["match", "fan_excitement", "media_velocity", "rivalry_score", "drama_score", "trending_keywords", "normalized_nlp_score"]].to_string(index=False))

print("\n✅ Pipeline de PLN y entrenamiento del clasificador completados y validados.")

--- RANKING DE PARTIDOS POR SCORE CUALITATIVO DE PLN [0-100] ---
     match  fan_excitement  media_velocity  rivalry_score  drama_score              trending_keywords  normalized_nlp_score
LIV vs EVE           55.18           81.67          0.633        0.733 merseyside, liverpool, everton                 67.05
TOT vs CHE           56.75           74.17          0.534        0.792       vs, chelsea, postecoglou                 64.92
AVL vs WOL           71.29           40.00          0.642        0.706           villa, wolves, derby                 61.91
MCI vs ARS           64.19          100.00          0.000        0.704         premier, league, title                 61.55
BHA vs CRY           55.81           32.50          0.531        0.681           brighton, vs, palace                 51.90
MUN vs FUL           38.01           54.17          0.000        0.583       united, fulham, trafford                 38.51
BOU vs SHU           55.66           18.33          0.186        0.